
# Procesamiento de la Carta Marina de Córdoba 2017

## Descripción

Este notebook tiene como objetivo reconstruir y documentar el proceso de extracción de datos de la Carta Marina de Córdoba correspondiente a las elecciones de 2017.

El desarrollo se basa en el trabajo realizado en el repositorio **Carta Marina Córdoba 2017** de avdata99 (https://github.com/avdata99/carta-marina-2017/tree/master).

En esta etapa el trabajo se concentra exclusivamente en la Carta Marina 2017.

## Flujo de trabajo


1. PDF
2. TXT (pdftotext -layout)
3. CSV de mesas/escuelas/electores
4. Geolocalización de escuelas
5. CSV geolocalizado
6. Mapas y análisis


**Productos esperados**

- `LugaresDeVotacion-elecciones-2015.pdf`
- `carta-marina-cordoba-2015.txt`
- `escuelas-elecciones-2015-cordoba.csv`
- `escuelas-elecciones-2015-cordoba-clean.csv` (si requiere correcciones)



0. Dependencias

In [1]:
from google.colab import files
import pandas as pd
from pathlib import Path

In [2]:
!apt-get update -qq
!apt-get install -y -qq poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up libpoppler118:amd64 (22.02.0-2ubuntu0.13) ...
Setting up poppler-

#1. Carga del PDF

In [3]:
archivo = files.upload()

nombre_archivo = next(iter(archivo))

print(f"Archivo cargado: {nombre_archivo}")


Saving 2017.pdf to 2017.pdf
Archivo cargado: 2017.pdf


#2. PDF a TXT

In [4]:
nombre_txt = Path(nombre_archivo).with_suffix(".txt")

!pdftotext -layout "{nombre_archivo}" "{nombre_txt}"

print(f"Archivo generado: {nombre_txt}")

Archivo generado: 2017.txt


#3. TXT a CSV



```
Leer línea
    │
    ├── ¿Es un encabezado o número de página?
    │      └── Sí → ignorar
    │
    ├── ¿Es una sección?
    │      └── Sí → actualizar sección
    │
    ├── ¿Es un circuito?
    │      └── Sí → actualizar circuito e iniciar lectura de establecimientos
    │
    ├── ¿Es el resumen del circuito?
    │      └── Sí → finalizar lectura del circuito
    │
    ├── ¿Está vacía?
    │      └── Sí → ignorar
    │
    ├── ¿Estamos dentro de un circuito?
    │      └── No → continuar
    │
    └── Procesar establecimiento
           ├── Ignorar líneas "asociada a"
           ├── Reconstruir el nombre del establecimiento
           ├── Extraer mesas y electores
           ├── Validar continuidad de mesas
           └── Guardar registro
```





##Inspeccion del TXT

In [5]:
path = f"/content/{nombre_txt}"

with open(path, "r", encoding="utf-8") as f:
    raw = f.read()

lines = raw.splitlines()

print(f"Cantidad total de líneas: {len(lines)}")
print("\nPrimeras 10 líneas:\n")

for i, linea in enumerate(lines[:10], start=1):
    print(i, repr(linea))
print ("...")
print ("...")
print ("...")
print ("...")
for i, linea in enumerate(raw.splitlines()[-10:], start=len(raw.splitlines())-9):
    print(i, repr(linea))

Cantidad total de líneas: 3951

Primeras 10 líneas:

1 '                                        DISTRITO CORDOBA'
2 '                                           ELECCIONES 2017'
3 '                           Informe de Establecimientos,Mesas y Electores Habilitados'
4 ''
5 ''
6 ''
7 ''
8 'Sección 1 - CAPITAL'
9 ''
10 ' Circuito   1 - SECCIONAL PRIMERA                                   Cant Mesas   Mesas desde / hasta Cant Electores'
...
...
...
...
3942 '                  RESUMEN DEL DISTRITO:                 CORDOBA'
3943 '                  Establecimientos Habilitados:                   1.211'
3944 '                  Mesas Habilitadas:                              8.649'
3945 '                  Electores Habilitados:                      2.884.358'
3946 ''
3947 ''
3948 ''
3949 ''
3950 '                                                                                              Página 70 de 70'
3951 ''


In [6]:
print("Saltos de línea \\n:", raw.count("\n"))
print("Saltos de página \\f:", raw.count("\f"))

print("Con split('\\n'):", len(raw.split("\n")))
print("Con splitlines():", len(raw.splitlines()))

Saltos de línea \n: 3881
Saltos de página \f: 70
Con split('\n'): 3882
Con splitlines(): 3951


##Inicialización de variables

In [26]:
# =====================================
# Inicialización de variables del parser
# =====================================

# Sección electoral (equivale al departamento de la provincia)
seccion_nro = 0
seccion_name = ""

# Circuito electoral dentro de la sección.
# Puede contener letras (ej.: 4A, 4B, 4C), por eso se almacena como texto.
circuito_nro = ""
circuito_name = ""

# Estado actual del parser.
# Cuando vale "escuelas", las líneas leídas corresponden a establecimientos.
imin = ""

# Contador de líneas procesadas del archivo TXT.
cnt = 0

# Contador de errores (heredado del parser original).
errores = 0

# Lista donde se almacenan todos los establecimientos extraídos.
# Cada elemento es un diccionario con la información de un establecimiento.
escuelas = []

# Registra las discontinuidades detectadas en la numeración de mesas.
# Se utiliza como control de calidad, pero no interrumpe la ejecución.
discontinuidades = []

# Última mesa procesada.
# Permite verificar que la numeración de mesas sea continua.
last_mesa = 0

# ============================
# Variables de diagnóstico
# ============================

# Cantidad de números de página ignorados.
paginas_ignoradas = 0

# Cantidad de encabezados "DISTRITO CORDOBA" ignorados.
distritos_ignorados = 0

# Cantidad de encabezados "ELECCIONES 2015" ignorados.
elecciones_ignoradas = 0

# Cantidad de encabezados "Informe de Establecimientos" ignorados.
informes_ignorados = 0

# Almacena las filas que no pudieron procesarse correctamente,
# junto con la información necesaria para su revisión.
filas_con_error = []

print(f"Contador inicial: {cnt}")


Contador inicial: 0


##Ciclo for

In [27]:
for linea in lines:
    cnt += 1

    # Ignorar número de página
    if "Página" in linea:
        paginas_ignoradas += 1
        continue

    # Ignorar encabezados
    if linea.strip() == "DISTRITO CORDOBA":
        distritos_ignorados += 1
        continue

    if "ELECCIONES 20" in linea:
        elecciones_ignoradas += 1
        continue

    if "Informe de Establecimientos" in linea:
        informes_ignorados += 1
        continue

    # Detectar sección
    if linea.startswith("Sección "):
        contenido = linea.removeprefix("Sección ").strip()
        partes = contenido.split("-", 1)

        seccion_nro = int(partes[0].strip())
        seccion_name = partes[1].strip()

        imin = ""
        continue

    # Detectar circuito
    if linea.startswith(" Circuito"):
        contenido = linea.removeprefix(" Circuito").strip()
        partes = contenido.split("-", 1)

        circuito_nro = partes[0].strip()

        # En la misma línea aparecen los títulos de otras columnas
        partes_nombre = partes[1].split("       ")
        circuito_name = partes_nombre[0].strip()

        imin = "escuelas"
        continue

    # Detectar el final del listado de escuelas
    # Las líneas vacías se ignoran, pero no cierran el circuito
    if linea == "":
       continue

    # Solo el resumen cierra realmente el circuito
    if linea.startswith("Resúmen del Circuito"):
        imin = ""
        continue

    # Procesar escuelas
    if imin == "escuelas":
        partes = linea.split("    ")

        # Conservar solo fragmentos con contenido
        datos = [x.strip() for x in partes if x.strip() != ""]

        print(f"{cnt} -- {datos}")

        if len(datos) == 1 and "asociada a" in datos[0]:
            continue



        establecimiento = " ".join(datos[:-3])
        cant_mesas = int(datos[-3])

        rango_mesas = datos[-2].split(" a ")
        mesa_desde = int(rango_mesas[0])
        mesa_hasta = int(rango_mesas[1])


        if mesa_desde != last_mesa + 1:
            discontinuidades.append({
              "linea": cnt,
              "esperada": last_mesa + 1,
              "encontrada": mesa_desde,
              "establecimiento": establecimiento
          })
            #raise ValueError(
                #f"Mesa inválida en línea {cnt}: "
                #f"empieza en {mesa_desde}, "
                #f"pero la anterior terminó en {last_mesa}"
            #)

        last_mesa = mesa_hasta

        cant_electores = int(datos[-1].replace(".", ""))

        elem = {
            "seccion_nro": seccion_nro,
            "seccion_name": seccion_name,
            "circuito_nro": circuito_nro,
            "circuito_name": circuito_name,
            "establecimiento": establecimiento.replace(",", "."),
            "cant_mesas": cant_mesas,
            "desde": mesa_desde,
            "hasta": mesa_hasta,
            "electores": cant_electores,
        }

        escuelas.append(elem)
        print(f"Contador inicial: {cnt}")

11 -- ['CENTRO EDUC.NIVEL MEDIO ADULTO - DEAN FUNES 417', '7', '00001 a 00007', '2.401']
Contador inicial: 11
12 -- ['ESC NUESTRA SEÑORA DEL HUERTO - BELGRANO 269', '12', '00008 a 00019', '4.116']
Contador inicial: 12
13 -- ['COL NAC DE MONSERRAT - OBISPO TREJO 294', '18', '00020 a 00037', '6.158']
Contador inicial: 13
14 -- ['ESC SANTA TERESA DE JESUS - OBISPO TREJO Y SANABRIA 160', '10', '00038 a 00047', '3.420']
Contador inicial: 14
18 -- ['ESC JUAN BAUTISTA ALBERDI - GRAL PAZ 486', '18', '00048 a 00065', '6.145']
Contador inicial: 18
22 -- ['ESC JERONIMO LUIS DE CABRERA - SANTA ROSA 650', '15', '00066 a 00080', '5.175']
Contador inicial: 22
23 -- ['IPEM N° 270 GRAL M BELGRANO - DEAN FUNES 850', '11', '00081 a 00091', '3.795']
Contador inicial: 23
24 -- ['ESC SANTO TOMAS - CASEROS 745', '8', '00092 a 00099', '2.760']
Contador inicial: 24
25 -- ['ESC NORMAL ALEJANDRO CARBO - AV COLON 951', '13', '00100 a 00112', '4.474']
Contador inicial: 25
26 -- ['ESC MARIANO MORENO - SANTA ROSA ES

In [28]:
print(discontinuidades)
print("Establecimientos:", len(escuelas))
print("Discontinuidades:", len(discontinuidades))
print("Última mesa:", last_mesa)
for d in discontinuidades:
    print(d)

[{'linea': 2320, 'esperada': 6000, 'encontrada': 6001, 'establecimiento': 'ESC NICOLAS AVELLANEDA - VICENTE LOPEZ 538'}, {'linea': 2391, 'esperada': 6339, 'encontrada': 6000, 'establecimiento': 'ESCUELA N°359 - WASHINGTON'}, {'linea': 3325, 'esperada': 7780, 'encontrada': 7788, 'establecimiento': 'ESCUELA JOSE MARIA PAZ - LOS ANDES(B°CAMARA) 80'}, {'linea': 3326, 'esperada': 7797, 'encontrada': 7780, 'establecimiento': 'ESCUELA MANUEL SOLARES - ESPAÑA 26'}, {'linea': 3327, 'esperada': 7788, 'encontrada': 7797, 'establecimiento': 'ESC SANTIAGO DE LINIERS - B°LINIERS'}]
Establecimientos: 1211
Discontinuidades: 5
Última mesa: 8649
{'linea': 2320, 'esperada': 6000, 'encontrada': 6001, 'establecimiento': 'ESC NICOLAS AVELLANEDA - VICENTE LOPEZ 538'}
{'linea': 2391, 'esperada': 6339, 'encontrada': 6000, 'establecimiento': 'ESCUELA N°359 - WASHINGTON'}
{'linea': 3325, 'esperada': 7780, 'encontrada': 7788, 'establecimiento': 'ESCUELA JOSE MARIA PAZ - LOS ANDES(B°CAMARA) 80'}
{'linea': 3326, 'e

In [25]:
print("Electores:", sum(e["electores"] for e in escuelas))

Electores: 2884358


# Diagnostico de filas problematicas por su nombre y domicilio
para despues limpar a mano

In [29]:
# ============================================================
# DIAGNÓSTICO INDEPENDIENTE DE FILAS PROBLEMÁTICAS
# No modifica las variables del parser definitivo
# ============================================================

modo_diag = ""
linea_diag = 0
ultima_mesa_diag = 0

filas_correctas_diag = 0
filas_problematicas = []
discontinuidades_diag = []

for linea in lines:
    linea_diag += 1

    # Ignorar encabezados y elementos que no son escuelas
    if "Página" in linea:
        continue

    if linea.strip() == "DISTRITO CORDOBA":
        continue

    if "ELECCIONES 20" in linea:
        continue

    if "Informe de Establecimientos" in linea:
        continue

    if linea.startswith("Sección "):
        modo_diag = ""
        continue

    if linea.startswith(" Circuito"):
        modo_diag = "escuelas"
        continue

    if linea == "":
        continue

    if linea.startswith("Resúmen del Circuito"):
        modo_diag = ""
        continue

    # Solo analizar líneas que deberían contener escuelas
    if modo_diag == "escuelas":
        partes = linea.split("    ")
        datos = [x.strip() for x in partes if x.strip() != ""]

        try:
            # Se prueba exactamente la estructura del parser original
            escuela = datos[0]
            cant_mesas = int(datos[1])

            rango_mesas = datos[2].split(" a ")
            mesa_desde = int(rango_mesas[0])
            mesa_hasta = int(rango_mesas[1])

            cant_electores = int(datos[3].replace(".", ""))

            # Registrar discontinuidades, pero sin detener el diagnóstico
            if mesa_desde != ultima_mesa_diag + 1:
                discontinuidades_diag.append({
                    "linea": linea_diag,
                    "esperada": ultima_mesa_diag + 1,
                    "encontrada": mesa_desde,
                    "contenido": linea,
                })

            ultima_mesa_diag = mesa_hasta
            filas_correctas_diag += 1

        except Exception as error:
            filas_problematicas.append({
                "linea": linea_diag,
                "contenido": linea,
                "datos": datos,
                "error": str(error),
            })

            print(f"Fila problemática en línea {linea_diag}")
            print(f"Datos obtenidos: {datos}")
            print(f"Error: {error}")
            print("-" * 80)

            # Intentar recuperar solamente el rango de mesas para que
            # el diagnóstico pueda continuar con la numeración correcta
            for fragmento in datos:
                if " a " in fragmento:
                    try:
                        rango_auxiliar = fragmento.split(" a ")
                        mesa_desde_aux = int(rango_auxiliar[0])
                        mesa_hasta_aux = int(rango_auxiliar[1])

                        if mesa_desde_aux != ultima_mesa_diag + 1:
                            discontinuidades_diag.append({
                                "linea": linea_diag,
                                "esperada": ultima_mesa_diag + 1,
                                "encontrada": mesa_desde_aux,
                                "contenido": linea,
                            })

                        ultima_mesa_diag = mesa_hasta_aux
                        break

                    except ValueError:
                        pass

            continue

Fila problemática en línea 216
Datos obtenidos: ['ESC PETER PAN - P DE GUZMAN', 'B°M DE SOBREMONTE 131', '15', '00802 a 00816', '5.198']
Error: invalid literal for int() with base 10: 'B°M DE SOBREMONTE 131'
--------------------------------------------------------------------------------
Fila problemática en línea 218
Datos obtenidos: ['JOSE LUIS SERCIC - PEREZ CORREA', 'M DE SOBREMONTE 1.500', '8', '00820 a 00827', '2.768']
Error: invalid literal for int() with base 10: 'M DE SOBREMONTE 1.500'
--------------------------------------------------------------------------------
Fila problemática en línea 234
Datos obtenidos: ['ESC NACIONES UNIDAS - CTDA', '8 S/N', '9', '00845 a 00853', '3.096']
Error: invalid literal for int() with base 10: '8 S/N'
--------------------------------------------------------------------------------
Fila problemática en línea 241
Datos obtenidos: ['ESC FRANCISCO VIDAL - FRAGUEIRO', '(MANZANA "C") 4.200', '12', '00878 a 00889', '4.062']
Error: invalid literal fo

In [30]:
print("Filas procesadas correctamente:", filas_correctas_diag)
print("Filas problemáticas:", len(filas_problematicas))
print("Discontinuidades detectadas:", len(discontinuidades_diag))
print("Última mesa encontrada:", ultima_mesa_diag)

Filas procesadas correctamente: 1164
Filas problemáticas: 51
Discontinuidades detectadas: 5
Última mesa encontrada: 8649


In [19]:
for fila in filas_problematicas:
    print("Línea:", fila["linea"])
    print("Contenido:", fila["contenido"])
    print("Separación obtenida:", fila["datos"])
    print("Error:", fila["error"])
    print("-" * 100)

Línea: 216
Contenido: ESC PETER PAN - P DE GUZMAN    B°M DE SOBREMONTE 131                             15    00802 a 00816             5.198
Separación obtenida: ['ESC PETER PAN - P DE GUZMAN', 'B°M DE SOBREMONTE 131', '15', '00802 a 00816', '5.198']
Error: invalid literal for int() with base 10: 'B°M DE SOBREMONTE 131'
----------------------------------------------------------------------------------------------------
Línea: 218
Contenido: JOSE LUIS SERCIC - PEREZ CORREA    M DE SOBREMONTE 1.500                          8    00820 a 00827             2.768
Separación obtenida: ['JOSE LUIS SERCIC - PEREZ CORREA', 'M DE SOBREMONTE 1.500', '8', '00820 a 00827', '2.768']
Error: invalid literal for int() with base 10: 'M DE SOBREMONTE 1.500'
----------------------------------------------------------------------------------------------------
Línea: 234
Contenido: ESC NACIONES UNIDAS - CTDA    8 S/N                                            9    00845 a 00853             3.096
Separación ob

#Dataframe

In [31]:
df_escuelas = pd.DataFrame(escuelas)

display(df_escuelas.head())
display(df_escuelas.tail())

print(df_escuelas.shape)

,seccion_nro,seccion_name,circuito_nro,circuito_name,establecimiento,cant_mesas,desde,hasta,electores
0,1,CAPITAL,1,SECCIONAL PRIMERA,CENTRO EDUC.NIVEL MEDIO ADULTO - DEAN FUNES 417,7,1,7,2401
1,1,CAPITAL,1,SECCIONAL PRIMERA,ESC NUESTRA SEÑORA DEL HUERTO - BELGRANO 269,12,8,19,4116
2,1,CAPITAL,1,SECCIONAL PRIMERA,COL NAC DE MONSERRAT - OBISPO TREJO 294,18,20,37,6158
3,1,CAPITAL,1,SECCIONAL PRIMERA,ESC SANTA TERESA DE JESUS - OBISPO TREJO Y SAN...,10,38,47,3420
4,1,CAPITAL,2,SECCIONAL SEGUNDA,ESC JUAN BAUTISTA ALBERDI - GRAL PAZ 486,18,48,65,6145


,seccion_nro,seccion_name,circuito_nro,circuito_name,establecimiento,cant_mesas,desde,hasta,electores
1206,26,UNION,397A,CORRAL DEL BAJO,ESCUELA VALENTIN ALSINA - CORRAL DEL BAJO,1,8634,8634,7
1207,26,UNION,400,SAN MARCOS SUD,INST JOSE DE SAN MARTIN - SAN MARCOS SUD,5,8635,8639,1570
1208,26,UNION,400,SAN MARCOS SUD,ESC PROV M BUCHARDO - SAN MARCOS SUD,4,8640,8643,1253
1209,26,UNION,401,SANTA MARIA,ESC PROV PAULA ALBARRACIN - PUBLICA S/N,1,8644,8644,226
1210,26,UNION,402,VIAMONTE,INSTITUTO JUAN B ALBERDI - AVELLANEDA 182,5,8645,8649,1599


(1211, 9)


In [ ]:
df_escuelas.sample(20, random_state=42)

,seccion_nro,seccion_name,circuito_nro,circuito_name,establecimiento,cant_mesas,desde,hasta,electores
319,1,CAPITAL,14F,GRANJA DE FUNES,ESC JAVIER LAZCANO COLODRERO - CARLOS BROGGI S...,9,2989,2997,3015
956,20,SAN JUSTO,281,BEIRO,ESCUELA N°349(ESC JUAN BEIRO) - BEIRO,1,7039,7039,155
1181,26,UNION,400,SAN MARCOS SUD,INST JOSE DE SAN MARTIN - SAN MARCOS SUD,4,8398,8401,1380
86,1,CAPITAL,7G,PANAMERICANO,"ESC FRANCISCO VIDAL - FRAGUEIRO (MANZANA ""C""...",12,857,868,4097
990,20,SAN JUSTO,302,MAUNIER,ESC PROV GRAL SAN MARTIN - PUBLICA S/N,1,7214,7214,113
458,4,CRUZ DEL EJE,45,EL BRETE,ESC PROV MANUEL BELGRANO - PUBLICA S/N,4,4030,4033,1172
1035,21,SANTA MARIA,316,ALTA GRACIA,ESC SANTIAGO DE LINIERS - B°LINIERS,8,7590,7597,2744
946,20,SAN JUSTO,278,ALTOS DE CHIPION,ESC.PROV.RECONQUISTA - SARMIENTO 543,6,6956,6961,1853
323,1,CAPITAL,14H,MERCANTIL,ESC ARTEMIO ARAN - PUBLICA ESQ JACINTO YABEN,6,3025,3030,1968
361,2,CALAMUCHITA,20A,VILLA GENERAL BELGRANO,ESC.GRAL J. DE SAN MARTIN - VILLA GRAL.BELGRANO,9,3234,3242,3069


In [32]:
output_path = "escuelas-elecciones-2017-cordoba.csv"

df_escuelas.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivo exportado: {output_path}")

Archivo exportado: escuelas-elecciones-2015-cordoba.csv
